# Lesson 01 - AI 代理介紹

歡迎來到 **AI 代理初學者** 課程的第一課！

**AI 代理** 是一個使用大型語言模型 (LLM) 作為其推理引擎的程式，並且可以在現實世界中採取*行動*——呼叫 API、查詢資料庫或執行程式碼——以代表使用者完成目標。

在本筆記本中，你將建立第一個代理：一個推薦假期目的地的**旅遊代理**。在此過程中，你將學習如何：

1. 使用 **Microsoft Agent Framework** 連接至 Azure AI Foundry Agent 服務。
2. 給代理一個**工具**——一個可以呼叫的普通 Python 函式。
3. 執行代理並檢視其回應。
4. 逐字元串流代理的回應。


## Setup

在執行此筆記本之前，請確保您已：

1. **擁有一個 Azure AI Foundry 專案** 並部署了聊天模型（例如 `gpt-4o-mini`）。
2. **透過 Azure CLI 登入** — 在終端機執行 `az login`。
3. **設置必要的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署的模型名稱。

以下的儲存格會安裝您需要的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 建立你的第一個代理人

一個代理人需要兩樣東西：

- **指令**，告訴它*是誰*以及*如何行為*（系統提示）。
- **工具** — 使用 `@tool` 裝飾的 Python 函數，代理人可以調用這些函數來獲取資訊或執行操作。

以下我們定義一個簡單的工具，回傳熱門旅遊目的地的清單。當使用者請求旅遊建議時，代理人會使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在本課程中，您學會了如何：

- **建立一個 provider**，透過 `AzureAIProjectAgentProvider` 連接到 Azure AI Foundry Agent 服務。
- **使用 `@tool` 裝飾器定義工具**，使代理能夠調用您的 Python 函數。
- **運行代理**，並使用使用者訊息並列印其回應。
- **串流回應**，以實現即時輸出。

在下一課中，我們將更深入探索代理架構，並學習如何賦予代理更強大的工具和多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於關鍵資料，建議採用專業人工翻譯。我們對因使用本翻譯而引起的任何誤解或誤釋概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
